# Bird Identification using CV - Part 2: Segmentation

**ITAI 1378  |  Midterm  |  2026**

**Group 8**

**Author:** Stuart Fairchild | Kalen Foster | Ranveer Chand

---

## Segmentation

Expanding the findings from the proof of concept, we'll use YOLO11Large to detect birds in the scene and SAM to provide segmentation masks

In [1]:
%pip install -q ultralytics

from ultralytics import YOLO, SAM
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

print("Setup complete. Ready to detect and segment.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Setup complete. Ready to detect and segment.


---

## Load images from Google Drive

Connect to Google Drive to load test image directory.

To use this without changes, add images to Google Drive under

`/Colab Notebooks/ITAI1378/midterm/images`

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
base_path = '/content/drive/MyDrive/Colab Notebooks/ITAI1378/midterm'
images_path = '/content/drive/MyDrive/Colab Notebooks/ITAI1378/midterm/images'

for filename in os.listdir(images_path):
  if filename.endswith((".png", ".jpg")):
    file_path = os.path.join(images_path, filename)
    print(f"Current file: {filename}")
    # Uncomment below if you want to load all images for viewing
    # img = Image.open(file_path)
    # plt.figure(figsize=(8, 10))
    # plt.imshow(img)
    # plt.axis("off")
    # plt.show()

Current file: 20260711_110012851062.jpg
Current file: 20260711_110607700821.jpg
Current file: 20260711_110614124173.jpg
Current file: 20260711_110623390691.jpg
Current file: 20260711_114504335390.jpg
Current file: 20260711_114510046678.jpg
Current file: 20260711_114518126954.jpg
Current file: 20260711_114652178971.jpg
Current file: 20260711_114656013898.jpg
Current file: 20260711_122042601055.jpg
Current file: 20260711_122047486200.jpg
Current file: 20260711_122057774137.jpg
Current file: 20260711_124529347193.jpg
Current file: 20260711_125021249691.jpg
Current file: 20260711_125032098571.jpg
Current file: 20260711_125043301951.jpg
Current file: 20260711_131423814430.png
Current file: 20260711_131457505871.png
Current file: 20260711_163118393903.png
Current file: 20260711_163140593762.png
Current file: 20260711_163151124444.png
Current file: 20260711_163202648535.png
Current file: 20260711_163215955185.png
Current file: 20260711_163224382030.png
Current file: 20260711_195811947191.png


## 7. Running detection with YOLO11 Large and segmenting with SAM2


In [8]:
model = "YOLO11Large_SAM2"
detector = YOLO("yolo11l.pt")
print(f"{model} loaded.")
sam = SAM("sam2.1_s.pt")

detected_path = f"{base_path}/detected_{model}"
for filename in os.listdir(images_path):
  if filename.endswith((".png", ".jpg")):
    file_path = os.path.join(images_path, filename)
    print(f"Current file: {filename}")
    results = detector.predict(
        source=file_path,
        conf=0.12,           # Set confidence threshold lower
        classes=[14]        # 14 is the COCO index for 'bird'
    )
    # annotated = results[0].plot()
    # annotated_rgb = annotated[..., ::-1]
    # plt.figure(figsize=(10, 12))
    # plt.imshow(annotated_rgb)
    # plt.axis("off")
    # plt.title(f"{model} detection results: {filename}")
    # output_file = f"{detected_path}/{filename}"
    # print(f"Output detected file: {output_file}")
    # plt.savefig(output_file)
    # plt.show()
    # SAM2 segmentation from the YOLO11 output
    yolo_boxes = results[0].boxes.xyxy.cpu().numpy()

    # Only run SAM if YOLO detected any birds
    if len(yolo_boxes) > 0:
      sam_results = sam(file_path, bboxes=yolo_boxes)
      annotated = sam_results[0].plot()[..., ::-1]
      plt.figure(figsize=(10, 12))
      plt.imshow(annotated)
      plt.axis("off")
      plt.title(f"{model}: {filename}")
      output_file = f"{detected_path}/{filename}"
      print(f"Output detected file: {output_file}")
      plt.savefig(output_file)
      plt.show()
    else:
      print(f"No birds detected in {filename}. Skipping SAM segmentation.")

Output hidden; open in https://colab.research.google.com to view.